# 26_cited_refs: reference lists of cited works outside the pulled set (pilot test, 2026-11-12)

Thin driver: logic lives in `citecurv.data.cited`. Runtime: **CPU** (no GPU needed), Python 3.12 or newer.

Why: the S2.2 sweep on the pilot (PROGRESS.md) showed that the Forman screen is nearly constant there. The
screen is F = 2 - in_deg_t(citing) - out_deg_t(cited), and the cited work's term is its reference count; a pull
stores reference lists only for the works that match the slice filter, so the term was zero for every cited
work outside it (98.95 % of the pilot's events). The Ollivier/coupling part needs the reference ids themselves.
This notebook fetches, by id, `referenced_works_count` and `referenced_works` for those cited works.

Reads from `MyDrive/citecurv/<RUN>/`: `events.parquet`, `works.parquet`.
Writes there: `cited_refs.parquet` (work_id, pub_date, n_refs, ref_ids), `cited_refs_stats.json`,
`cited_refs_record.json`. Needs the Colab secret `OPENALEX_API_KEY`. The pull resumes after a disconnect: rerun all cells.

Cost: one OpenAlex list call per 100 ids (10 credits each on a free key, 100,000 credits a day).


In [ ]:
# ---- Python version gate -----------------------------------------------------------------------------
import platform
import sys

print("python  ", sys.version)
if sys.version_info < (3, 12):
    raise RuntimeError(f"This notebook needs Python 3.12 or newer, found {platform.python_version()}.")


In [ ]:
# ---- install citecurv with the data-stage pins (configs/pins.md) ------------------------------------------
import subprocess

REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
REPO_TAG = "main"   # pilot test; set a release tag for a run that goes into the results
subprocess.run(["rm", "-rf", "/content/repo"], check=True)   # a rerun clones again
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_TAG, REPO, "/content/repo"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
%pip install -q /content/repo "pyalex==0.21" pyarrow requests pyyaml

import importlib.metadata as md

VERSIONS = {"python": sys.version.split()[0], "commit": COMMIT}
for pkg in ("citecurv", "pyalex", "numpy", "pandas", "pyarrow"):
    VERSIONS[pkg] = md.version(pkg)
print(VERSIONS)


In [ ]:
# ---- Drive, API key (Colab Secrets), settings ------------------------------------------------------------
import json
import os
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["OPENALEX_API_KEY"] = userdata.get("OPENALEX_API_KEY")   # never print or save it

from citecurv.config import load_config

cfg = load_config("/content/repo/configs/default.yaml")
RUN = "pilot"                                   # same value as in 10_data
RUN_DIR = Path("/content/drive/MyDrive") / "citecurv" / RUN
EVENTS = RUN_DIR / "events.parquet"
WORKS = RUN_DIR / "works.parquet"
CITED_REFS = RUN_DIR / "cited_refs.parquet"
assert EVENTS.exists() and WORKS.exists(), f"run 10_data with RUN = {RUN!r} first"
print(RUN_DIR)


In [ ]:
# ---- which cited works have no record ----------------------------------------------------------------------
import pandas as pd

from citecurv.data.cited import cited_only_ids
from citecurv.data.events import read_events

events = read_events(EVENTS)
works = pd.read_parquet(WORKS, columns=["work_id"])
ids = cited_only_ids(events, works)
print(f"{len(events):,} events | {events['dst_id'].nunique():,} cited works | {len(ids):,} of them without a record")


In [ ]:
# ---- OpenAlex credits left today, and what this pull needs ---------------------------------------------------
import math

import requests

r = requests.get("https://api.openalex.org/rate-limit", params={"api_key": os.environ["OPENALEX_API_KEY"]}, timeout=30)
print(r.status_code, (r.json() if r.ok else {}).get("rate_limit"))
print("calls needed:", math.ceil(len(ids) / 100), "(10 credits each on a free key)")


In [ ]:
# ---- fetch reference count and reference ids by id (resumable: finished parts are kept in cited_refs_parts/) ----
import time

from citecurv.data.cited import pull_works_by_ids
from citecurv.data.openalex import BudgetExhausted

ID_FILTER = "openalex"   # found by the live probe of 25_cited_text on 2026-11-12
t0 = time.perf_counter()
try:
    pull_stats = pull_works_by_ids(ids, CITED_REFS, config=cfg, id_filter=ID_FILTER, what="references", log=print)
except BudgetExhausted as e:
    print(e)   # reset time (UTC); finished parts are saved, rerun this notebook after the reset
    raise
pull_stats["minutes"] = round((time.perf_counter() - t0) / 60, 2)
print(pull_stats)


In [ ]:
# ---- what came back, and how much of the event table it covers -------------------------------------------------
refs = pd.read_parquet(CITED_REFS)
n_refs = refs.set_index("work_id")["n_refs"]
outside = ~events["dst_id"].isin(works["work_id"])
got = events["dst_id"].map(n_refs)
record = {
    "run": RUN, "versions": VERSIONS, "pull": pull_stats,
    "n_events": int(len(events)), "n_events_cited_outside": int(outside.sum()),
    "n_events_cited_outside_with_count": int((outside & got.notna()).sum()),
    "n_events_cited_outside_count_zero": int((outside & got.eq(0)).sum()),
    "n_refs_quantiles": {str(q): float(v) for q, v in refs["n_refs"].quantile([0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).items()},
    "file_mb": round(CITED_REFS.stat().st_size / 2**20, 1),
}
(RUN_DIR / "cited_refs_record.json").write_text(json.dumps(record, indent=1))
print(json.dumps(record, indent=1))
